INSTALL DEPENDENCIES

In [4]:
!pip install -q datasets==2.19.1
!pip install -q trl==0.8.6
!pip install peft 
!pip install -q transformers==4.41.0
!pip install -q bitsandbytes==0.43.1
!pip install -q sentencepiece==0.1.99
!pip install -q accelerate==0.30.1
!pip install -q huggingface_hub==0.23.1


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 1.0 MB/s  0:00:0075.1 kB/s eta 0:00:01

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [45 lines of output]
      Checking for Rust toolchain....
      Rust not found, installing into a temporary directory
      Python reports SOABI: cpython-313-darwin
      Computed rustc target triple: aarch64-apple-darwin
      Installation directory: /Users/mohammadehsan/Library/Caches/puccinialin
      Rustup already downloaded
      Installing rust to /Users/mohammadehsan/Library/Caches/puccinialin/rustup
      warn: It looks

In [2]:
!pip uninstall -y bitsandbytes triton
!pip install bitsandbytes

  Obtaining dependency information for bitsandbytes from https://files.pythonhosted.org/packages/8b/6c/b3c2a6b05e0fb06c6258b675436b2b090192450d7f283826e6323471e270/bitsandbytes-0.50.0-py3-none-macosx_14_0_arm64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.2/123.2 kB 154.9 kB/s eta 0:00:001m148.1 kB/s eta 0:00:01

[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


IMPORT LIBRARIES

In [22]:
import torch 
from transformers import AutoModelForCausalLM , AutoTokenizer,BitsAndBytesConfig,TrainingArguments,Trainer,DataCollatorForLanguageModeling,GenerationConfig
from peft import LoraConfig , PeftModel ,get_peft_model
from trl import DPOTrainer ,DPOConfig 
import bitsandbytes as bnb 
from getpass import getpass
from datasets import load_dataset

In [2]:
def load_model_and_tokenizer(model_name:str):
    model=AutoModelForCausalLM.from_pretrained(model_name)
    tokenizer=AutoTokenizer.from_pretrained(model_name)
    return model , tokenizer 


model , tokenizer=load_model_and_tokenizer("./model/qwen2.5:0.5b:instruct")


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 6798.24it/s]


In [3]:
def load_dataset_func(dataset_name:str,split:str="train"):
    dataset=load_dataset(dataset_name)
    return dataset[split]



dataset=load_dataset_func("./dataset")

dataset




Dataset({
    features: ['system', 'question', 'chosen', 'rejected'],
    num_rows: 12859
})

In [4]:
dataset_filtered=dataset.filter(lambda x:len(x["system"])<500 and 
                                len(x["question"])<500
                                and
                                len(x["chosen"])<500
                                and
                                len(x["rejected"])<500)
print(dataset_filtered)

Dataset({
    features: ['system', 'question', 'chosen', 'rejected'],
    num_rows: 1431
})


In [5]:
def chatml_format(example):
    prompt={"role":"system","content":example["system"]},
    {"role":"user","content":example["question"]}

    chosen={"role":"assistant","content":example["chosen"]}
    rejected={"role":"assistant","content":example["rejected"]}


    return {
        "prompt":prompt,
        "chosen":chosen,
        "rejected":rejected
    }











In [6]:
dataset_filtered=dataset_filtered.map(
    chatml_format,
    remove_columns=dataset_filtered.column_names
)


In [7]:
dataset_filtered

Dataset({
    features: ['chosen', 'rejected', 'prompt'],
    num_rows: 1431
})

In [8]:
dashline="-".join("" for _ in range(20))
print(dataset_filtered["chosen"][0])
print(dashline)
print(dataset_filtered["rejected"][0])
print(dashline)
print(dataset_filtered["prompt"][0])

{'role': 'assistant', 'content': 'Midsummer House is a moderately priced Chinese restaurant with a 3/5 customer rating, located near All Bar One.'}
-------------------
{'role': 'assistant', 'content': ' Sure! Here\'s a sentence that describes all the data you provided:\n\n"Midsummer House is a moderately priced Chinese restaurant with a customer rating of 3 out of 5, located near All Bar One, offering a variety of delicious dishes."'}
-------------------
[{'role': 'system', 'content': 'You are an AI assistant. You will be given a task. You must generate a detailed and long answer.'}]


In [9]:
def apply_chat_template_format(example):
    prompt=tokenizer.apply_chat_template(example["prompt"],tokenize=False,add_generation_prompt=True)
    chosen=example["chosen"]["content"]
    rejected=example["rejected"]["content"]

    return {
        "prompt":prompt,
        "chosen":chosen,
        "rejected":rejected
    }



In [10]:
dataset=dataset_filtered.map(
    apply_chat_template_format
)

In [11]:
lora_config=LoraConfig(
    r=8,
    target_modules="all-linear",
    lora_alpha=8,
    lora_dropout=0.01,
    bias="none",
    task_type="CAUSAL_LM",

)


In [12]:
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16

)

In [13]:
model=AutoModelForCausalLM.from_pretrained("./model/qwen2.5:0.5b:instruct",quantization_config=bnb_config)
model

Loading weights: 100%|██████████| 290/290 [00:02<00:00, 140.26it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=896, out_features=896, bias=True)
          (k_proj): Linear4bit(in_features=896, out_features=128, bias=True)
          (v_proj): Linear4bit(in_features=896, out_features=128, bias=True)
          (o_proj): Linear4bit(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear4bit(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear4bit(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e

In [14]:
model_peft=get_peft_model(model,lora_config)


<h2>START TRAINING </h2>

In [15]:
def Training_args_func(output_dir:str,per_device_train_batch_size:int,per_device_eval_batch_size:int,num_train_epochs:int,lr:int,gradient_accumulation_steps:int,optim:str,warmup_steps:int):
    training_args=DPOConfig(
        output_dir=output_dir,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_eval_batch_size,
        num_train_epochs=num_train_epochs,
        learning_rate=lr,
        gradient_accumulation_steps=gradient_accumulation_steps,
        gradient_checkpointing=True,
        remove_unused_columns=False,
        save_strategy="epoch",
        lr_scheduler_type="cosine",
        logging_steps=1,
        optim=optim,
        warmup_steps=warmup_steps,
        bf16=True,
        report_to="none",


    )

    return training_args


training_args=Training_args_func(output_dir="./logs",per_device_train_batch_size=4,per_device_eval_batch_size=4,num_train_epochs=3,lr=5.0e-06,gradient_accumulation_steps=2,optim="paged_adamw_32bit",warmup_steps=2)



In [149]:
def Trainer_func(model,args,tokenizer,dataset):
    trainer=DPOTrainer(
        model=model,
        args=args,
        processing_class=tokenizer,
        train_dataset=dataset,
    )

    return trainer


trainer=Trainer_func(model_peft,training_args,tokenizer,dataset)
trainer.train()

Tokenizing train dataset: 100%|██████████| 1431/1431 [00:00<00:00, 1495.51 examples/s]
Dropping fully truncated examples from train dataset: 100%|██████████| 1431/1431 [00:00<00:00, 16003.16 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/mohammadehsan/Desktop/dpo-finetuning-orca-pairs/transformer_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [20]:
print(dataset["prompt"][1])
print(dataset["chosen"][1])

<|im_start|>system
You are an AI assistant. You will be given a task. You must generate a detailed and long answer.<|im_end|>
<|im_start|>assistant

A crash occurred on Fife Street early Saturday morning, prompting police to appeal for witnesses and close York Road, causing diversions.


<h2>GET INFRENCE </h2>

In [21]:
prompt="""
"3713841893836/4?\nLimit your response to mathematical expressions and symbols.
"""
system_prompt={"role":"user","content":prompt}


In [ ]:
model_peft=PeftModel.from_pretrained(model=model,config=)
tokenizer=AutoTokenizer.from_pretrained(config=)



In [ ]:
from transformers import pipeline
generator=pipeline(
    task="text-generation",
    model=model_peft
    tokenizer=tokenizer
)
model_kwargs_config=GenerationConfig(
    do_sample=True,
    temperature=0.7,
    top_p=0.3,
    num_return_sequences=1,
    max_length=200,

)
generated_text=generator(system_prompt,model_kwargs_config)
print(generated_text)